# MLP Optimisation Comparison and Confirmation

This notebook compares Random Search and GA under equal 40-unique-evaluation budgets. After both HPO runs exist, it can run three-seed confirmation and save the final selected hyperparameters. It does not use the frozen test set.


## 1. Package Setup


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/"
            "MLAAD_Scan_MFCC_Analysis_cache.ipynb, then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


required_cache_files = [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]
for required in required_cache_files:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Keep these names for older checklist cells, but they now point to cache metadata copied
# from the single canonical shared manifest rather than a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Load Shared MLP-Ready Cache


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/MLAAD_Scan_MFCC_Analysis_cache.ipynb, "
            "then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Compatibility aliases for older checklist cells. These are cache metadata copied from the
# single canonical shared manifest, not a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 3. Shared MLP Search Space and Training Helpers


In [ ]:
SEARCH_SPACE = {
    "hidden_units": [
        [64],
        [128],
        [64, 32],
        [128, 64],
        [256, 128],
        [256, 128, 64],
    ],
    "activation": ["relu", "tanh"],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.4],
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "batch_size": [32, 64, 128, 256],
}
SEARCH_KEYS = ["hidden_units", "activation", "dropout", "learning_rate", "batch_size"]
EXPECTED_SEARCH_SPACE_SIZE = 1200


def normalise_config(config):
    return {
        "hidden_units": [int(value) for value in config["hidden_units"]],
        "activation": str(config["activation"]),
        "dropout": float(config["dropout"]),
        "learning_rate": float(config["learning_rate"]),
        "batch_size": int(config["batch_size"]),
    }


def enumerate_search_space():
    configs = []
    for values in product(*(SEARCH_SPACE[key] for key in SEARCH_KEYS)):
        configs.append(normalise_config(dict(zip(SEARCH_KEYS, values))))
    return configs


ALL_CONFIGURATIONS = enumerate_search_space()
if len(ALL_CONFIGURATIONS) != EXPECTED_SEARCH_SPACE_SIZE:
    raise RuntimeError(f"Expected 1200 configurations, found {len(ALL_CONFIGURATIONS)}")


def config_json(config):
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def build_mlp_model(config, input_dim):
    config = normalise_config(config)
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for units in config["hidden_units"]:
        model.add(Dense(units, activation=config["activation"]))
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=config["learning_rate"])
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


def flatten_config(config):
    config = normalise_config(config)
    return {
        "hidden_units": json.dumps(config["hidden_units"]),
        "activation": config["activation"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "config_json": config_json(config),
        "config_key": config_key(config),
    }


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_mlp_model(config, X_train.shape[1])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime_seconds = time.perf_counter() - start_time

    validation_probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    validation_pred = (validation_probability >= THRESHOLD).astype(int)
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_val_loss = float(np.min(history.history["val_loss"]))

    return {
        "run_name": run_name,
        "seed": int(run_seed),
        "validation_macro_f1": f1_score(y_validation, validation_pred, average="macro", zero_division=0),
        "validation_binary_f1_synthetic": f1_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_accuracy": accuracy_score(y_validation, validation_pred),
        "validation_precision_synthetic": precision_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_recall_synthetic": recall_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_trained": int(len(history.history["loss"])),
        "early_stopped": bool(len(history.history["loss"]) < MAX_EPOCHS),
        "runtime_seconds": float(runtime_seconds),
        "objective": "validation_macro_f1",
        "threshold": THRESHOLD,
        **flatten_config(config),
    }


print("Shared HPO search-space size:", len(ALL_CONFIGURATIONS))


## 4. Optimisation Comparison and Three-Seed Confirmation


In [ ]:
RUN_CONFIRMATION = False
VERBOSE_TRAINING = 2
CONFIRMATION_SEEDS = [42, 123, 2026]

random_search_trials_path = HPO_DIR / "random_search_trials.csv"
genetic_algorithm_trials_path = HPO_DIR / "genetic_algorithm_trials.csv"
comparison_path = HPO_DIR / "optimisation_comparison.csv"
confirmation_results_path = HPO_DIR / "confirmation_results.csv"
confirmation_summary_path = HPO_DIR / "confirmation_summary.csv"
selected_hyperparameters_path = MODELS_DIR / "selected_hyperparameters.json"
convergence_plot_path = FIGURES_DIR / "mlp_hpo_convergence.png"


def load_trials(path, method):
    if not path.exists():
        print(f"Missing {method} trials: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["method"] = method
    return df


def unique_evaluations_only(df):
    if df.empty:
        return df
    if "cache_hit" not in df.columns:
        return df.copy()
    return df.loc[df["cache_hit"].astype(str).str.lower() == "false"].copy()


def summarise_trials(df, method):
    unique_df = unique_evaluations_only(df)
    if unique_df.empty:
        return None
    best = unique_df.sort_values("validation_macro_f1", ascending=False).iloc[0]
    return {
        "method": method,
        "best_validation_macro_f1": float(best["validation_macro_f1"]),
        "best_validation_loss": float(best["best_val_loss"]),
        "best_configuration": best["config_json"],
        "unique_evaluations": int(unique_df["config_key"].nunique()),
        "evaluation_number_where_best_found": int(best["unique_evaluation_number"]),
        "total_search_runtime_seconds": float(unique_df["runtime_seconds"].sum()),
    }


random_trials = load_trials(random_search_trials_path, "random_search")
ga_trials = load_trials(genetic_algorithm_trials_path, "genetic_algorithm")
comparison_rows = [
    row
    for row in [
        summarise_trials(random_trials, "random_search"),
        summarise_trials(ga_trials, "genetic_algorithm"),
    ]
    if row
]

if comparison_rows:
    optimisation_comparison_df = pd.DataFrame(comparison_rows)
    optimisation_comparison_df.to_csv(comparison_path, index=False)
    display(optimisation_comparison_df)

    plt.figure(figsize=(8, 5))
    for label, df in [("Random Search", random_trials), ("Genetic Algorithm", ga_trials)]:
        unique_df = unique_evaluations_only(df)
        if unique_df.empty:
            continue
        unique_df = unique_df.sort_values("unique_evaluation_number")
        unique_df["best_macro_f1_so_far"] = unique_df["validation_macro_f1"].cummax()
        plt.plot(unique_df["unique_evaluation_number"], unique_df["best_macro_f1_so_far"], marker="o", label=label)
    plt.xlabel("Unique model evaluations")
    plt.ylabel("Best validation macro-F1 so far")
    plt.title("MLP HPO convergence")
    plt.legend()
    plt.tight_layout()
    plt.savefig(convergence_plot_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Comparison requires Random Search and GA trial files.")


def candidate_configs_from_hpo(random_df, ga_df):
    candidates = []
    seen = set()
    for method, df in [("random_search", random_df), ("genetic_algorithm", ga_df)]:
        unique_df = unique_evaluations_only(df)
        if unique_df.empty:
            continue
        best = unique_df.sort_values("validation_macro_f1", ascending=False).iloc[0]
        if best["config_key"] not in seen:
            candidates.append({"source_hpo_method": method, "config": json.loads(best["config_json"]), "config_key": best["config_key"]})
            seen.add(best["config_key"])

    non_empty = [df for df in [random_df, ga_df] if not df.empty]
    combined = pd.concat(non_empty, ignore_index=True) if non_empty else pd.DataFrame()
    if not combined.empty:
        for _, row in unique_evaluations_only(combined).sort_values("validation_macro_f1", ascending=False).head(3).iterrows():
            if row["config_key"] not in seen:
                candidates.append({"source_hpo_method": row["method"], "config": json.loads(row["config_json"]), "config_key": row["config_key"]})
                seen.add(row["config_key"])
    return candidates


def run_confirmation():
    candidates = candidate_configs_from_hpo(random_trials, ga_trials)
    if not candidates:
        raise RuntimeError("No HPO candidates are available for confirmation.")

    rows = []
    for candidate_id, candidate in enumerate(candidates, start=1):
        for seed in CONFIRMATION_SEEDS:
            row = train_and_evaluate_config(
                candidate["config"],
                run_seed=seed,
                run_name=f"confirmation_candidate_{candidate_id}_seed_{seed}",
                verbose=VERBOSE_TRAINING,
            )
            row.update(
                {
                    "candidate_id": candidate_id,
                    "source_hpo_method": candidate["source_hpo_method"],
                    "confirmation_seed": seed,
                }
            )
            rows.append(row)
            pd.DataFrame(rows).to_csv(confirmation_results_path, index=False)

    confirmation_df = pd.DataFrame(rows)
    summary_df = (
        confirmation_df.groupby(["candidate_id", "source_hpo_method", "config_key", "config_json"], as_index=False)
        .agg(
            mean_validation_macro_f1=("validation_macro_f1", "mean"),
            std_validation_macro_f1=("validation_macro_f1", "std"),
            mean_validation_loss=("best_val_loss", "mean"),
        )
        .sort_values(
            ["mean_validation_macro_f1", "std_validation_macro_f1", "mean_validation_loss"],
            ascending=[False, True, True],
        )
    )
    summary_df.to_csv(confirmation_summary_path, index=False)
    selected = summary_df.iloc[0]
    selected_payload = {
        "configuration": json.loads(selected["config_json"]),
        "source_hpo_method": selected["source_hpo_method"],
        "mean_validation_macro_f1": float(selected["mean_validation_macro_f1"]),
        "std_validation_macro_f1": float(selected["std_validation_macro_f1"]),
        "mean_validation_loss": float(selected["mean_validation_loss"]),
        "confirmation_seeds": CONFIRMATION_SEEDS,
        "selection_rule": "highest mean validation macro-F1; lower standard deviation and validation loss as secondary evidence",
    }
    with open(selected_hyperparameters_path, "w", encoding="utf-8") as f:
        json.dump(selected_payload, f, indent=2)
    return confirmation_df, summary_df, selected_payload


if RUN_CONFIRMATION:
    confirmation_results_df, confirmation_summary_df, selected_hyperparameters = run_confirmation()
    display(confirmation_summary_df)
elif selected_hyperparameters_path.exists():
    print("RUN_CONFIRMATION is False. Existing selected hyperparameters are available:")
    with open(selected_hyperparameters_path, "r", encoding="utf-8") as f:
        display(pd.DataFrame([json.load(f)]))
else:
    print("RUN_CONFIRMATION is False. Set it to True after Random Search and GA have completed.")


## 5. Reproducibility Checklist


In [ ]:
checks = {
    "comparison_budget": "40 unique evaluations per HPO method",
    "confirmation_seeds": [42, 123, 2026],
    "selection_metric": "mean validation macro-F1",
    "test_set_loaded": False,
    "selected_hyperparameters_file": str(MODELS_DIR / "selected_hyperparameters.json"),
    "next_notebook": "04_MLP_Final_Selected_Model.ipynb",
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
